In [24]:
import pandas as pd
import glob

# Kraken2 rank → standard rank
rank_map = {
    "D": "Domain",
    "P": "Phylum",
    "C": "Class",
    "O": "Order",
    "F": "Family",
    "G": "Genus",
    "S": "Species"
}

# Thresholds
MIN_READS = 10       # minimum reads to consider a taxon
MAJORITY = 0.51      # 80% majority to call unambiguous

# Get all Kraken2 report files
files = glob.glob("Kraken2-Report/*.fasta.tabular")

all_summaries = []

for f in files:
    df = pd.read_csv(f, sep="\t", header=None,
                     names=["percent","reads_total","reads_direct","rank","taxid","name"])
    
    # Keep only ranks we care about and filter low reads
    df = df[df['rank'].isin(rank_map.keys())].copy()
    df = df[df['reads_total'] >= MIN_READS]
    df['rank_name'] = df['rank'].map(rank_map)
    
    summary_dict = {}
    
    for rank in rank_map.values():
        rank_df = df[df['rank_name'] == rank]
        
        if rank_df.empty:
            summary_dict[rank] = ""
            continue
        
        # Compute majority fraction
        grouped = rank_df.groupby('name')['reads_total'].sum()
        top_name = grouped.idxmax().strip()  # trim spaces
        top_reads = grouped.max()
        total_reads = grouped.sum()
        
        if top_reads / total_reads >= MAJORITY:
            summary_dict[rank] = top_name
        else:
            summary_dict[rank] = ""  # ambiguous
    
    summary_dict['MAG'] = f.split("/")[-1].replace(".fasta.tabular","")
    
    # If species is Homo sapiens, blank all taxa
    if summary_dict.get("Species","").lower() == "homo sapiens":
        for rank in rank_map.values():
            summary_dict[rank] = ""
    
    all_summaries.append(summary_dict)

# Build final dataframe
final_df = pd.DataFrame(all_summaries)

# Reorder columns: MAG first
cols = ['MAG', "Domain","Phylum","Class","Order","Family","Genus","Species"]
final_df = final_df[cols]

# Sort hierarchically by taxonomy columns
taxonomy_cols = ["Domain","Phylum","Class","Order","Family","Genus","Species"]
final_df_sorted = final_df.sort_values(by=taxonomy_cols, na_position='last')

# Save sorted tab-delimited file
final_df_sorted.to_csv("MAGs_taxonomy_summary_sorted.tsv", sep="\t", index=False)

print(final_df_sorted.head())


                   MAG Domain Phylum Class Order Family Genus Species
1  ERR9966629_bin_4769                                               
3   ERR9966624_bin_404                                               
5  ERR9966620_bin_1002                                               
7    ERR9966620_bin_31                                               
8    ERR9966626_bin_22                                               
